# QMSS Project — The Work-From-Home Shift
**Department of HSS | Quantitative Methods for Social Sciences**

*Did Remote Work Change People's Eating, Activity, and Mental Health?*



## 1. Imports, Data Loading & Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import chi2, f as f_dist

# ─── Load ───
df = pd.read_csv("Impact_of_Remote_Work_on_Mental_Health.csv")
df.columns = df.columns.str.strip()
print("Original shape:", df.shape)
print("\nColumn list:")
print(df.columns.tolist())
df.head()


In [ ]:
# ─── Clean: drop rows missing key analysis variables ───
ANALYSIS_VARS = ['Work_Location', 'Stress_Level', 'Sleep_Quality',
                 'Physical_Activity', 'Work_Life_Balance_Rating']

df_clean = df.dropna(subset=ANALYSIS_VARS).copy()
print(f"Records after cleaning: {len(df_clean):,} / {len(df):,}")
print("\nMissing values in cleaned dataset:")
print(df_clean[ANALYSIS_VARS].isnull().sum())


## 2. Descriptive Statistics

In [ ]:
# ─── 2.1  Sample distribution by Work Location ───
dist = df_clean['Work_Location'].value_counts().reset_index()
dist.columns = ['Work_Location', 'Count']
dist['Percentage (%)'] = (dist['Count'] / dist['Count'].sum() * 100).round(2)
print(dist.to_string(index=False))

# Pie chart
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(dist['Count'], labels=dist['Work_Location'],
       autopct='%1.1f%%', startangle=140,
       colors=['#4C72B0','#DD8452','#55A868'])
ax.set_title('Figure 3.1 – Sample Distribution by Work Location', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("fig3_1_sample_dist.png", dpi=150)
plt.show()


In [ ]:
# ─── 2.2  Key descriptive stats by Work Location ───
desc = df_clean.groupby('Work_Location').agg(
    n=('Employee_ID', 'count'),
    Avg_Hours_Worked=('Hours_Worked_Per_Week', lambda x: round(x.mean(), 2)),
    Avg_WLB_Rating=('Work_Life_Balance_Rating', lambda x: round(x.mean(), 3)),
    Avg_Social_Isolation=('Social_Isolation_Rating', lambda x: round(x.mean(), 3))
).reset_index()
print(desc.to_string(index=False))


In [ ]:
# ─── 2.3  Hours Worked Per Week – Box Plot ───
groups = [df_clean[df_clean['Work_Location'] == loc]['Hours_Worked_Per_Week'].dropna()
          for loc in ['Hybrid', 'Onsite', 'Remote']]

fig, ax = plt.subplots(figsize=(7, 4))
bp = ax.boxplot(groups, patch_artist=True, labels=['Hybrid', 'Onsite', 'Remote'],
                medianprops=dict(color='black', linewidth=2))
colors = ['#4C72B0','#DD8452','#55A868']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_title('Figure 3.6 – Hours Worked Per Week by Work Location', fontsize=11, fontweight='bold')
ax.set_ylabel('Hours / Week')
plt.tight_layout()
plt.savefig("fig3_6_hours_boxplot.png", dpi=150)
plt.show()


## 3. Manual Chi-Square Test Function

> All three categorical hypotheses (H1, H3, Sleep Quality) use this function which computes observed frequencies, expected frequencies, the χ² statistic, degrees of freedom, and p-value step-by-step without relying on scipy's chi2_contingency.

In [ ]:
def manual_chi_square(data, row_var, col_var, hypothesis_label=""):
    """
    Computes Chi-Square Test of Independence manually.
    Prints full step-by-step workings.
    Returns (chi2_stat, dof, p_value, observed, expected).
    """
    print(f"{'='*60}")
    if hypothesis_label:
        print(f"  {hypothesis_label}")
    print(f"  Chi-Square Test: {row_var} vs {col_var}")
    print(f"{'='*60}")

    # Step 1 – Observed
    observed = pd.crosstab(data[row_var], data[col_var])
    print("\nStep 1 – Observed Frequencies (O):")
    print(observed.to_string())

    # Step 2 – Marginals
    row_totals = observed.sum(axis=1)
    col_totals  = observed.sum(axis=0)
    N           = observed.values.sum()
    print(f"\nStep 2 – Row Totals:\n{row_totals.to_string()}")
    print(f"\n         Column Totals:\n{col_totals.to_string()}")
    print(f"\n         Grand Total N = {N}")

    # Step 3 – Expected
    expected = pd.DataFrame(
        np.outer(row_totals.values, col_totals.values) / N,
        index=observed.index, columns=observed.columns
    )
    print("\nStep 3 – Expected Frequencies E = (Row × Col) / N:")
    print(expected.round(4).to_string())

    # Step 4 – (O-E)²/E table
    contrib = ((observed - expected) ** 2) / expected
    print("\nStep 4 – Chi-Square Contributions (O−E)²/E:")
    print(contrib.round(4).to_string())

    # Step 5 – χ²
    chi2_stat = contrib.values.sum()
    r, c = observed.shape
    dof = (r - 1) * (c - 1)
    print(f"\nStep 5 – χ² statistic = {chi2_stat:.4f}")
    print(f"         Degrees of freedom df = ({r}-1)×({c}-1) = {dof}")

    # Step 6 – p-value
    p_val = 1 - chi2.cdf(chi2_stat, dof)
    alpha = 0.05
    decision = ("Reject H₀  → Significant association ✅"
                if p_val < alpha else
                "Fail to Reject H₀ → No significant association ❌")
    print(f"\nStep 6 – p-value = {p_val:.4f}  (α = {alpha})")
    print(f"         Decision : {decision}")

    return chi2_stat, dof, p_val, observed, expected


## 4. H1 — Physical Activity vs Work Location (Chi-Square)

In [ ]:
chi2_h1, dof_h1, p_h1, obs_h1, exp_h1 = manual_chi_square(
    df_clean, 'Work_Location', 'Physical_Activity',
    "H1: Daily physical activity is significantly lower in WFH (Remote) workers"
)


In [ ]:
# ─── Bar Chart: Physical Activity vs Work Location ───
freq_h1 = pd.crosstab(df_clean['Work_Location'], df_clean['Physical_Activity'])
pct_h1  = freq_h1.div(freq_h1.sum(axis=1), axis=0) * 100

ax = pct_h1.plot(kind='bar', figsize=(7, 4), color=['#4C72B0','#DD8452'],
                 edgecolor='black', width=0.65)
ax.set_title('Figure 3.2 – Physical Activity Frequency by Work Location (%)', fontsize=11, fontweight='bold')
ax.set_xlabel('Work Location')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(pct_h1.index, rotation=0)
ax.legend(title='Activity')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%",
                (p.get_x() + p.get_width()/2, p.get_height() + 0.4),
                ha='center', fontsize=8)
plt.tight_layout()
plt.savefig("fig3_2_activity_bar.png", dpi=150)
plt.show()


## 5. H3 — Stress Level vs Work Location (Chi-Square)

In [ ]:
chi2_h3, dof_h3, p_h3, obs_h3, exp_h3 = manual_chi_square(
    df_clean, 'Work_Location', 'Stress_Level',
    "H3: Stress levels are significantly associated with work location type"
)


In [ ]:
# ─── Bar Chart: Stress Level vs Work Location ───
freq_h3 = pd.crosstab(df_clean['Work_Location'], df_clean['Stress_Level'])
pct_h3  = freq_h3.div(freq_h3.sum(axis=1), axis=0) * 100

order = ['Low','Medium','High']
pct_h3 = pct_h3[order]

ax = pct_h3.plot(kind='bar', figsize=(7, 4),
                 color=['#55A868','#FFC107','#DD8452'],
                 edgecolor='black', width=0.65)
ax.set_title('Figure 3.3 – Stress Level Distribution by Work Location (%)', fontsize=11, fontweight='bold')
ax.set_xlabel('Work Location')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(pct_h3.index, rotation=0)
ax.legend(title='Stress Level')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%",
                (p.get_x() + p.get_width()/2, p.get_height() + 0.4),
                ha='center', fontsize=8)
plt.tight_layout()
plt.savefig("fig3_3_stress_bar.png", dpi=150)
plt.show()


## 6. Sleep Quality vs Work Location (Chi-Square)

In [ ]:
chi2_sl, dof_sl, p_sl, obs_sl, exp_sl = manual_chi_square(
    df_clean, 'Work_Location', 'Sleep_Quality',
    "Sleep Quality — additional Chi-Square analysis"
)


In [ ]:
# ─── Bar Chart: Sleep Quality vs Work Location ───
freq_sl = pd.crosstab(df_clean['Work_Location'], df_clean['Sleep_Quality'])
pct_sl  = freq_sl.div(freq_sl.sum(axis=1), axis=0) * 100

order = ['Good','Average','Poor']
pct_sl = pct_sl[order]

ax = pct_sl.plot(kind='bar', figsize=(7, 4),
                 color=['#55A868','#FFC107','#C44E52'],
                 edgecolor='black', width=0.65)
ax.set_title('Figure 3.4 – Sleep Quality Distribution by Work Location (%)', fontsize=11, fontweight='bold')
ax.set_xlabel('Work Location')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(pct_sl.index, rotation=0)
ax.legend(title='Sleep Quality')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%",
                (p.get_x() + p.get_width()/2, p.get_height() + 0.4),
                ha='center', fontsize=8)
plt.tight_layout()
plt.savefig("fig3_4_sleep_bar.png", dpi=150)
plt.show()


## 7. H4 — Work-Life Balance Rating: One-Way ANOVA (Manual)

In [ ]:
# ─── Step-by-step One-Way ANOVA ───
print("="*60)
print("  H4: WLB Ratings differ significantly across Work Location groups")
print("  Test: One-Way ANOVA")
print("="*60)

groups_anova = {loc: grp['Work_Life_Balance_Rating'].values
                for loc, grp in df_clean.groupby('Work_Location')}

# Descriptives
print("\nStep 1 – Group Descriptives:")
desc_rows = []
for loc, vals in groups_anova.items():
    desc_rows.append({
        'Work_Location': loc,
        'n': len(vals),
        'Mean': round(vals.mean(), 4),
        'Std': round(vals.std(ddof=1), 4),
        'Min': vals.min(),
        'Max': vals.max()
    })
desc_df = pd.DataFrame(desc_rows)
print(desc_df.to_string(index=False))

# Grand mean
all_vals = np.concatenate(list(groups_anova.values()))
grand_mean = all_vals.mean()
N_total = len(all_vals)
k = len(groups_anova)
print(f"\nGrand Mean = {grand_mean:.4f},  N = {N_total},  k = {k}")

# SS Between
ss_between = sum(len(v) * (v.mean() - grand_mean)**2 for v in groups_anova.values())
df_between = k - 1
ms_between = ss_between / df_between
print(f"\nStep 2 – SS_between = {ss_between:.4f},  df_between = {df_between}")
print(f"          MS_between = {ms_between:.4f}")

# SS Within
ss_within = sum(((v - v.mean())**2).sum() for v in groups_anova.values())
df_within = N_total - k
ms_within = ss_within / df_within
print(f"\nStep 3 – SS_within  = {ss_within:.4f},  df_within = {df_within}")
print(f"          MS_within  = {ms_within:.4f}")

# F and p
F_stat = ms_between / ms_within
p_anova = 1 - f_dist.cdf(F_stat, df_between, df_within)
alpha = 0.05
decision = ("Reject H₀ → Significant difference ✅" if p_anova < alpha
            else "Fail to Reject H₀ → No significant difference ❌")
print(f"\nStep 4 – F-statistic = {F_stat:.4f}")
print(f"          p-value     = {p_anova:.4f}  (α = {alpha})")
print(f"          Decision    : {decision}")


In [ ]:
# ─── Box Plot: WLB Rating by Work Location ───
grp_data = [df_clean[df_clean['Work_Location'] == loc]['Work_Life_Balance_Rating'].dropna()
            for loc in ['Hybrid', 'Onsite', 'Remote']]

fig, ax = plt.subplots(figsize=(7, 4))
bp = ax.boxplot(grp_data, patch_artist=True,
                labels=['Hybrid', 'Onsite', 'Remote'],
                medianprops=dict(color='black', linewidth=2))
colors = ['#4C72B0','#DD8452','#55A868']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_title('Figure 3.5 – Work-Life Balance Rating by Work Location', fontsize=11, fontweight='bold')
ax.set_ylabel('WLB Rating (1 – 5)')
plt.tight_layout()
plt.savefig("fig3_5_wlb_boxplot.png", dpi=150)
plt.show()


## 8. Additional Finding — Satisfaction with Remote Work

In [ ]:
# ─── Satisfaction with Remote Work Arrangement ───
sat = pd.crosstab(df['Work_Location'], df['Satisfaction_with_Remote_Work'])
sat_pct = sat.div(sat.sum(axis=1), axis=0) * 100

ax = sat_pct.plot(kind='bar', figsize=(8, 4),
                  color=['#4C72B0','#55A868','#DD8452'],
                  edgecolor='black', width=0.65)
ax.set_title('Figure 3.7 – Satisfaction with Remote Work by Work Location (%)', fontsize=11, fontweight='bold')
ax.set_xlabel('Work Location')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(sat_pct.index, rotation=0)
ax.legend(title='Satisfaction')
plt.tight_layout()
plt.savefig("fig3_7_satisfaction_bar.png", dpi=150)
plt.show()


## 9. Export — Cumulative Analysis CSV

In [ ]:
rows = []

def add_ct(df_c, var, label):
    freq = pd.crosstab(df_c['Work_Location'], df_c[var])
    pct  = freq.div(freq.sum(axis=1), axis=0) * 100
    for loc in freq.index:
        for cat in freq.columns:
            rows.append({
                'Analysis':       label,
                'Work_Location':  loc,
                'Category':       cat,
                'Frequency':      int(freq.loc[loc, cat]),
                'Percentage':     round(pct.loc[loc, cat], 2)
            })

add_ct(df_clean, 'Physical_Activity', 'H1_Physical_Activity_vs_WorkLocation')
add_ct(df_clean, 'Stress_Level',      'H3_Stress_Level_vs_WorkLocation')
add_ct(df_clean, 'Sleep_Quality',     'Sleep_Quality_vs_WorkLocation')

# ANOVA descriptive rows
for loc, grp in df_clean.groupby('Work_Location'):
    vals = grp['Work_Life_Balance_Rating']
    rows.append({
        'Analysis':      'H4_WLB_ANOVA_Descriptive',
        'Work_Location':  loc,
        'Category':      f'n={len(vals)} | Mean={vals.mean():.3f} | SD={vals.std(ddof=1):.3f}',
        'Frequency':      len(vals),
        'Percentage':     round(vals.mean(), 4)
    })

# Hypothesis test summary rows
tests = [
    ('H1_Physical_Activity_vs_WorkLocation','Chi-Square','χ²=3.4537','df=2','p=0.1778','Fail to Reject H₀'),
    ('H3_Stress_Level_vs_WorkLocation',     'Chi-Square','χ²=4.1384','df=4','p=0.3876','Fail to Reject H₀'),
    ('Sleep_Quality_vs_WorkLocation',       'Chi-Square','χ²=5.9618','df=4','p=0.2020','Fail to Reject H₀'),
    ('H4_WLB_ANOVA_Descriptive',            'One-Way ANOVA','F=0.5032','dfB=2 dfW=3368','p=0.6046','Fail to Reject H₀'),
]
for analysis, test, stat, df_info, pval, decision in tests:
    rows.append({
        'Analysis':      f'SUMMARY_TEST_{analysis}',
        'Work_Location': 'ALL',
        'Category':      f'{test} | {stat} | {df_info} | {pval} | {decision}',
        'Frequency':     None,
        'Percentage':    None
    })

cum_df = pd.DataFrame(rows)
cum_df.to_csv("cumulative_analysis.csv", index=False)
print(f"Saved cumulative_analysis.csv  ({len(cum_df)} rows)")
cum_df


## 10. Final Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Hypothesis': 'H1', 'Variable': 'Physical Activity', 'Test': 'Chi-Square',
     'χ²/F': 3.4537, 'df': '2', 'p-value': 0.1778, 'Decision': 'Fail to Reject H₀'},
    {'Hypothesis': 'H3', 'Variable': 'Stress Level',      'Test': 'Chi-Square',
     'χ²/F': 4.1384, 'df': '4', 'p-value': 0.3876, 'Decision': 'Fail to Reject H₀'},
    {'Hypothesis': '—',  'Variable': 'Sleep Quality',     'Test': 'Chi-Square',
     'χ²/F': 5.9618, 'df': '4', 'p-value': 0.2020, 'Decision': 'Fail to Reject H₀'},
    {'Hypothesis': 'H4', 'Variable': 'Work-Life Balance', 'Test': 'One-Way ANOVA',
     'χ²/F': 0.5032, 'df': 'dfB=2, dfW=3368', 'p-value': 0.6046, 'Decision': 'Fail to Reject H₀'},
])
print("\n========= STATISTICAL TEST SUMMARY =========")
print(summary.to_string(index=False))
